# Traditional Object Detection

Methods: 1) Template Matching  2) Haar Cascade Face Detection  3) HOG + SVM Pedestrian Detection

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
%matplotlib inline

base = os.path.dirname(os.getcwd())

---
## Method 1: Template Matching

In [ ]:
path = os.path.join(base, 'IMG_1626.jpg')
if not os.path.exists(path):
    path = os.path.join(base, 'images.jpg')

img = cv2.imread(path)
if img is None:
    raise FileNotFoundError(f'Could not load image from {path}')

img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

h, w = gray.shape
template = gray[h//4:h//4+50, w//4:w//4+50].copy()

result = cv2.matchTemplate(gray, template, cv2.TM_CCOEFF_NORMED)
min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(result)
top_left = max_loc
h_t, w_t = template.shape
bottom_right = (top_left[0] + w_t, top_left[1] + h_t)
img_match = img_rgb.copy()
cv2.rectangle(img_match, top_left, bottom_right, (255, 0, 0), 3)

plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(template, cmap='gray')
plt.title('Template')
plt.axis('off')
plt.subplot(1, 3, 2)
plt.imshow(result, cmap='gray')
plt.title(f'Match Map (max={max_val:.2f})')
plt.axis('off')
plt.subplot(1, 3, 3)
plt.imshow(img_match)
plt.title('Best Match')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
threshold = 0.8
locations = np.where(result >= threshold)
img_multi = img_rgb.copy()
for pt in zip(*locations[::-1]):
    cv2.rectangle(img_multi, pt, (pt[0] + w_t, pt[1] + h_t), (0, 255, 0), 2)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(img_rgb)
plt.title('Original')
plt.axis('off')
plt.subplot(1, 2, 2)
plt.imshow(img_multi)
plt.title(f'Matches > {threshold}')
plt.axis('off')
plt.show()

---
## Method 2: Haar Cascade Face Detection

In [ ]:
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
if face_cascade.empty():
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_alt.xml')

path_face = os.path.join(base, 'IMG_7139.png')
if not os.path.exists(path_face):
    path_face = os.path.join(base, 'IMG_1626.jpg')
img_face = cv2.imread(path_face)

if img_face is None:
    print('Could not load image for face detection')
else:
    gray_face = cv2.cvtColor(img_face, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray_face, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
    img_face_rgb = cv2.cvtColor(img_face, cv2.COLOR_BGR2RGB)
    for (x, y, w, h) in faces:
        cv2.rectangle(img_face_rgb, (x, y), (x + w, y + h), (0, 255, 0), 3)

    plt.figure(figsize=(8, 6))
    plt.imshow(img_face_rgb)
    plt.title(f'Haar Cascade - Faces Detected: {len(faces)}')
    plt.axis('off')
    plt.show()
    if len(faces) == 0:
        print('No faces found. Try adjusting scaleFactor or minNeighbors.')

---
## Method 3: HOG + SVM Pedestrian Detection

In [ ]:
hog = cv2.HOGDescriptor()
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())

path_hog = os.path.join(base, 'IMG_1626.jpg')
if not os.path.exists(path_hog):
    path_hog = os.path.join(base, 'images.jpg')
img_hog = cv2.imread(path_hog)

if img_hog is None:
    print('Could not load image for HOG detection')
else:
    img_hog_rgb = cv2.cvtColor(img_hog, cv2.COLOR_BGR2RGB)
    (rects, _) = hog.detectMultiScale(img_hog, winStride=(4, 4), padding=(8, 8), scale=1.05)
    for (x, y, w, h) in rects:
        cv2.rectangle(img_hog_rgb, (x, y), (x + w, y + h), (0, 0, 255), 2)

    plt.figure(figsize=(10, 6))
    plt.imshow(img_hog_rgb)
    plt.title(f'HOG + SVM - Detections: {len(rects)}')
    plt.axis('off')
    plt.show()

In [ ]:
print('=' * 50)
print('Object Detection Methods Demonstrated:')
print('=' * 50)
print('1. Template Matching - cv2.matchTemplate()')
print('2. Haar Cascade - cv2.CascadeClassifier')
print('3. HOG + SVM - cv2.HOGDescriptor')
print('=' * 50)